# Identitas Penulis
- **Nama Lengkap**: Muhammad Ikctiar Saputra
- **NIM**: 250401020169
- **Kelas**: IF401
- **Program Studi**: PJJ Informatika

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Langkah 1: Generate & Eksplorasi Dataset Sintetis Harga Mobil
# Variabel:
# - umur_mobil (1-10 tahun)
# - jarak_tempuh_ribu_km (umur * 15 + noise)
# - tipe_bahan_bakar (0: Bensin, 1: Diesel)
# - harga_juta (target: dipengaruhi secara linear oleh umur, jarak, dan jenis bahan bakar)

np.random.seed(42)
n_samples = 200
umur = np.random.uniform(1, 10, n_samples)
jarak = umur * 15 + np.random.normal(0, 10, n_samples)
jarak = np.clip(jarak, 5, 200)
bahan_bakar = np.random.choice([0, 1], n_samples)  # 0: Bensin, 1: Diesel

# Formula harga mobil: base 300 juta - 15 juta * umur - 0.4 juta * jarak + 25 juta jika Diesel + noise
harga = 300 - 15 * umur - 0.4 * jarak + 25 * bahan_bakar + np.random.normal(0, 15, n_samples)

df_car = pd.DataFrame({
    'umur_mobil': umur,
    'jarak_tempuh_ribu_km': jarak,
    'tipe_bahan_bakar': bahan_bakar,
    'harga_juta': harga
})
print("Shape:", df_car.shape)
print(df_car.describe().round(2))

Shape: (200, 4)
       umur_mobil  jarak_tempuh_ribu_km  tipe_bahan_bakar  harga_juta
count      200.00                200.00            200.00      200.00
mean         5.28                 79.16              0.51      172.93
std          2.64                 41.22              0.50       51.98
min          1.03                  5.00              0.00       51.78
25%          2.90                 43.08              0.00      137.93
50%          5.18                 78.18              1.00      175.76
75%          7.51                111.45              1.00      212.87
max          9.96                181.82              1.00      283.82


In [2]:
# Langkah 2: Preprocessing
# Memisahkan fitur (X) dan target (y)
X = df_car.drop(columns=['harga_juta'])
y = df_car['harga_juta']

# Membagi data menjadi 80% Train dan 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (160, 3)
Test shape: (40, 3)


In [3]:
# Langkah 3: Latih Model & Tampilkan Koefisien
# Melatih algoritma Regresi Linier
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept (Konstanta):", np.round(model.intercept_, 2))
print("Koefisien (Bobot Fitur):")
for col, coef in zip(X.columns, model.coef_):
    print(f" - {col}: {np.round(coef, 2)}")

Intercept (Konstanta): 293.41
Koefisien (Bobot Fitur):
 - umur_mobil: -13.08
 - jarak_tempuh_ribu_km: -0.5
 - tipe_bahan_bakar: 23.36


In [4]:
# Langkah 4: Evaluasi Model
# Melakukan prediksi pada data pengujian dan mengukur kesalahan prediksi
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  (Mean Absolute Error): {mae:.2f} (dalam juta)")
print(f"MSE  (Mean Squared Error) : {mse:.2f}")
print(f"RMSE (Root Mean Sq. Error): {rmse:.2f} (dalam juta)")
print(f"R2   (R-squared)          : {r2:.4f}")

MAE  (Mean Absolute Error): 11.51 (dalam juta)
MSE  (Mean Squared Error) : 197.80
RMSE (Root Mean Sq. Error): 14.06 (dalam juta)
R2   (R-squared)          : 0.9234


### Interpretasi Metrik Evaluasi dalam Rupiah
Rata-rata kesalahan prediksi harga mobil yang dilakukan oleh model regresi adalah sebesar **Rp11.512.200** (didapatkan dari nilai MAE sebesar 11.51 dikali dengan 1.000.000 karena target harga bertipe data jutaan rupiah).
Nilai R-Squared (**0.9234**) mengindikasikan bahwa sebesar **92.34%** variasi harga mobil dalam dataset ini dapat dijelaskan dengan sangat baik oleh tiga fitur prediktor (umur mobil, jarak tempuh, dan tipe bahan bakar).

In [5]:
# Langkah 5: Visualisasi & Interpretasi Hasil Prediksi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted
sns.scatterplot(x=y_test, y=y_pred, ax=axes[0], color='blue', alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[0].set_title('Harga Aktual vs Harga Prediksi')
axes[0].set_xlabel('Harga Aktual (Juta IDR)')
axes[0].set_ylabel('Harga Prediksi (Juta IDR)')
axes[0].grid(True, alpha=0.3)

# Plot 2: Residual Plot
residuals = y_test - y_pred
sns.scatterplot(x=y_pred, y=residuals, ax=axes[1], color='red', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='--', lw=2)
axes[1].set_title('Residual Plot (Prediction Error)')
axes[1].set_xlabel('Harga Prediksi (Juta IDR)')
axes[1].set_ylabel('Residual (Error)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Kesimpulan
Pada pertemuan ketujuh ini, saya mempelajari konsep pembuatan dataset simulasi dan penerapan algoritma Regresi Linier:
1. Membuat dataset sintetis dengan relasi linear antar variabel menggunakan formula matematis yang mengandung bias dan noise.
2. Melatih model Regresi Linier (`LinearRegression`) dari scikit-learn dan mengekstrak bobot koefisien beserta konstanta intercept.
3. Mengevaluasi kualitas prediksi model menggunakan metrik standar seperti MAE, RMSE, dan R-Squared.
4. Membuat plot perbandingan Harga Aktual vs Prediksi untuk melihat keselarasan prediksi (semakin mendekati garis diagonal 45 derajat semakin baik).
5. Membuat Residual Plot untuk memeriksa apakah error terdistribusi secara acak di sekitar angka nol (*homoscedasticity*) yang menandakan model bekerja dengan baik tanpa bias struktural.

**Temuan Utama**: Umur mobil memiliki dampak negatif paling signifikan terhadap harga (harga turun sekitar Rp13.08 juta per tahun umur mobil). Sementara tipe bahan bakar Diesel mendongkrak harga jual sekitar Rp23.36 juta. Model ini sangat akurat dengan nilai R² sebesar 92.34%.

**Keterbatasan/Pertanyaan**: Hubungan antar variabel di dunia nyata seringkali bersifat non-linear. Bagaimana cara mengatasinya jika model regresi linier sederhana tidak lagi mampu menangkap hubungan non-linear tersebut?